# Experiment: AI-Augmented Idea Engine

This notebook prototypes a deterministic idea engine for wealth advisors. It scores the AI x India universe against different client profiles, generates short rationales backed by real metrics, and exports an internal memo to `reports/ai_idea_engine_notes.md`.

In [ ]:
from pathlib import Path
import logging
import sys

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'src').exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.analysis import build_ai_india_dataset, cache_stats_path
from src.reporting import archetype_output_table, build_idea_engine_markdown, write_markdown
from src.scoring import ARCHETYPE_CONFIGS, attach_rationales, compute_factor_scores, score_for_archetype

logging.basicConfig(level=logging.WARNING, format='%(levelname)s | %(message)s')
pd.set_option('display.max_columns', 50)


In [ ]:
stats_path = cache_stats_path()
if stats_path.exists():
    base_stats = pd.read_csv(stats_path)
    print(f'Loaded cached stats from {stats_path}')
else:
    _, _, _, base_stats = build_ai_india_dataset(start='2021-01-01')
    print('Cache not found, so the notebook pulled fresh live data.')

scored_stats = compute_factor_scores(base_stats)
scored_stats[['ticker', 'name', 'segment', 'return_1y', 'cagr_3y', 'annualized_volatility', 'dividend_yield', 'ai_purity_score']].head()


In [ ]:
archetype_summary = pd.DataFrame([
    {
        'Archetype': name,
        'Growth Priority': config['growth_priority'],
        'Risk Tolerance': config['risk_tolerance'],
        'Dividend Preference': config['dividend_preference'],
        'Concentration Preference': config['concentration_preference'],
        'AI Purity Preference': config['ai_purity_preference'],
        'Weights': ', '.join(f"{factor.replace('_score', '')}: {weight:.0%}" for factor, weight in config['weights'].items()),
    }
    for name, config in ARCHETYPE_CONFIGS.items()
])

archetype_summary


In [ ]:
display_columns = [
    'ticker',
    'name',
    'total_score',
    'growth_score',
    'momentum_score',
    'volatility_score',
    'yield_score',
    'ai_purity_score',
    'rationale_text',
]

archetype_tables = {}
for archetype_name, config in ARCHETYPE_CONFIGS.items():
    ranked = score_for_archetype(scored_stats, config)
    ranked = attach_rationales(ranked.head(7), archetype_name)
    display(Markdown(f'### {archetype_name}'))
    display(ranked[display_columns])
    archetype_tables[archetype_name] = archetype_output_table(ranked[display_columns])


In [ ]:
report_as_of = base_stats['as_of_date'].iloc[0] if 'as_of_date' in base_stats.columns else str(pd.Timestamp.today().date())

model_explanation = [
    'Growth score uses 3Y CAGR when available and otherwise falls back to 1Y return; scores are winsorized cross-sectionally to reduce single-name outlier effects.',
    'Momentum score blends 6M and 12M trailing performance so the model does not overreact to one short burst in price action.',
    'Volatility score rewards lower annualized volatility for conservative profiles, while aggressive profiles simply assign the factor a smaller weight.',
    'Yield score treats missing dividend data as neutral rather than inventing an income signal where the market-data feed is incomplete.',
    'AI purity is deterministic: AI infrastructure and analytics names score highest, engineering and vertical software names sit in the middle, and broad IT services score lower because AI is only part of the business mix.',
    'Short price histories are capped through a history penalty instead of being filled with made-up returns; this matters for newer listings such as Netweb or Tata Technologies.',
]

limitations = [
    'This is an educational prototype, not a suitability engine, optimizer, or recommendation system, and it should not be used as investment advice.',
    'The model only sees public price history, basic fundamentals, and a rule-based AI-purity signal; it does not incorporate earnings revisions, corporate actions review, management guidance, or compliance constraints.',
    'Public APIs can be incomplete or intermittently unavailable, so a real Morgan Stanley workflow would require licensed data feeds, content supervision, and auditable model governance.',
]

workflow_stories = [
    'Ahead of a meeting with a growth-oriented entrepreneur, an advisor can pull the Growth-Seeking and Aggressive Thematic lists, then focus the conversation on the difference between core IT compounders and higher-beta AI specialists.',
    'For a conservative HNI client, an advisor can start with the Defensive Income-Focused list, use the rationale text to shortlist steadier large-cap names, and then add one or two higher-purity AI satellites only if the client accepts the extra risk.',
]

report_text = build_idea_engine_markdown(
    as_of_date=report_as_of,
    model_explanation=model_explanation,
    limitations=limitations,
    archetype_tables=archetype_tables,
    workflow_stories=workflow_stories,
)

report_path = write_markdown('reports/ai_idea_engine_notes.md', report_text)
display(Markdown(f'Report written to `{report_path}`'))
display(Markdown(report_text[:2500]))
